# 🏆 2026 FIFA World Cup Predictor — Data Preprocessing

This notebook handles all data cleaning, filtering, and feature engineering for the World Cup prediction model.

**Pipeline:**
1. Load raw datasets (`results.csv`, `shootouts.csv`)
2. Explore & understand the data
3. Clean & filter for relevant matches
4. Engineer features for ML model
5. Export preprocessed dataset

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded successfully.")

## 2. Load Raw Data

We load two datasets:
- **results.csv** — 40,000+ international football matches since 1872
- **shootouts.csv** — Penalty shootout outcomes to adjust definitive winners

In [ ]:
# Load the datasets
results = pd.read_csv('results.csv')
shootouts = pd.read_csv('shootouts.csv')

print(f"Results dataset: {results.shape[0]:,} matches, {results.shape[1]} columns")
print(f"Shootouts dataset: {shootouts.shape[0]:,} records, {shootouts.shape[1]} columns")

## 3. Exploratory Data Analysis

In [ ]:
# Preview the results dataset
print("=" * 60)
print("RESULTS DATASET")
print("=" * 60)
display(results.head(10))
print(f"\nDate range: {results['date'].min()} → {results['date'].max()}")
print(f"Unique teams: {results['home_team'].nunique()}")
print(f"Unique tournaments: {results['tournament'].nunique()}")

In [ ]:
# Preview the shootouts dataset
print("=" * 60)
print("SHOOTOUTS DATASET")
print("=" * 60)
display(shootouts.head(10))

In [ ]:
# Data types and missing values
print("\n--- Results: Data Types & Nulls ---")
print(results.info())
print(f"\nMissing values:\n{results.isnull().sum()}")

In [ ]:
# Tournament distribution
print("\n--- Top 15 Tournaments by Match Count ---")
print(results['tournament'].value_counts().head(15))

## 4. Data Cleaning & Filtering

We filter to keep only **major competitive tournaments** that are most relevant for World Cup prediction.

In [ ]:
# Convert date column to datetime
results['date'] = pd.to_datetime(results['date'])

# Define major tournaments relevant to World Cup prediction
major_tournaments = [
    'FIFA World Cup',
    'FIFA World Cup qualification',
    'UEFA Euro',
    'UEFA Euro qualification',
    'Copa América',
    'African Cup of Nations',
    'AFC Asian Cup',
    'CONCACAF Gold Cup',
    'UEFA Nations League',
    'Confederations Cup',
]

# Filter for major tournaments only
df = results[results['tournament'].isin(major_tournaments)].copy()
print(f"Filtered from {results.shape[0]:,} → {df.shape[0]:,} matches (major tournaments only)")

In [ ]:
# Drop rows with missing values
before = df.shape[0]
df.dropna(inplace=True)
after = df.shape[0]
print(f"Dropped {before - after} rows with missing values. Remaining: {after:,}")

## 5. Penalty Shootout Adjustments

When a match ends in a draw but goes to penalties, we adjust the result to reflect the **definitive winner**.

In [ ]:
# Merge shootout winners into the main dataset
shootouts['date'] = pd.to_datetime(shootouts['date'])

df = df.merge(
    shootouts[['date', 'home_team', 'away_team', 'winner']],
    on=['date', 'home_team', 'away_team'],
    how='left',
    suffixes=('', '_shootout')
)

print(f"Matches with shootout data: {df['winner'].notna().sum()}")

## 6. Feature Engineering

We create features that capture each team's historical performance.

In [ ]:
# Determine match outcome from the home team's perspective
def get_match_result(row):
    """Returns 'Win', 'Loss', or 'Draw' from the home team's perspective."""
    if row['home_score'] > row['away_score']:
        return 'Win'
    elif row['home_score'] < row['away_score']:
        return 'Loss'
    else:
        # Check shootout winner for draws
        if pd.notna(row.get('winner')):
            return 'Win' if row['winner'] == row['home_team'] else 'Loss'
        return 'Draw'

df['result'] = df.apply(get_match_result, axis=1)

print("Match result distribution:")
print(df['result'].value_counts())

In [ ]:
# Goal difference
df['goal_diff'] = df['home_score'] - df['away_score']

# Year & month for temporal features
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

# Is it a neutral venue?
# (already present in the dataset as 'neutral' column if available)
if 'neutral' not in df.columns:
    df['neutral'] = False

print("\nNew features added: goal_diff, year, month")
display(df[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'goal_diff', 'result', 'neutral']].head(10))

In [ ]:
# Calculate rolling win rate for each team (last 10 matches)
def compute_team_stats(df, team_col, n=10):
    """Compute rolling win rate for teams."""
    team_stats = {}
    for team in df[team_col].unique():
        mask = (df['home_team'] == team) | (df['away_team'] == team)
        team_matches = df[mask].sort_values('date')
        
        wins = []
        for _, row in team_matches.iterrows():
            if row['home_team'] == team:
                wins.append(1 if row['result'] == 'Win' else 0)
            else:
                wins.append(1 if row['result'] == 'Loss' else 0)  # Away team wins when home loses
        
        # Rolling win rate over last n matches
        win_series = pd.Series(wins)
        rolling_wr = win_series.rolling(n, min_periods=1).mean()
        team_stats[team] = rolling_wr.iloc[-1] if len(rolling_wr) > 0 else 0.0
    
    return team_stats

home_stats = compute_team_stats(df, 'home_team')
away_stats = compute_team_stats(df, 'away_team')

df['home_win_rate'] = df['home_team'].map(home_stats)
df['away_win_rate'] = df['away_team'].map(away_stats)

print("✅ Rolling win rates computed for all teams.")
display(df[['home_team', 'away_team', 'home_win_rate', 'away_win_rate', 'result']].head(10))

## 7. Final Dataset Summary

In [ ]:
# Summary statistics
print("=" * 60)
print("PREPROCESSED DATASET SUMMARY")
print("=" * 60)
print(f"Total matches: {df.shape[0]:,}")
print(f"Features: {df.shape[1]}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Unique teams: {df['home_team'].nunique()}")
print(f"Tournaments: {df['tournament'].nunique()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nResult distribution:\n{df['result'].value_counts(normalize=True).round(3)}")

display(df.describe())

## 8. Export Preprocessed Data

In [ ]:
# Save the preprocessed dataset
output_path = 'preprocessed_matches.csv'
df.to_csv(output_path, index=False)
print(f"✅ Preprocessed data saved to '{output_path}' ({df.shape[0]:,} rows, {df.shape[1]} columns)")